In [ ]:
import pandas as pd

df = pd.read_csv("preprocessed_data.csv")

print("Dataset shape:", df.shape)
df.head()


In [ ]:
df.columns


In [ ]:
TARGET = "delivery_status"

X = df.drop(columns=[
    TARGET,
    "delivery_status_encoded"   # <-- THIS REMOVES LEAKAGE
])

y = df[TARGET]


In [ ]:
y.unique()


In [ ]:
label_map = {
    "On-Time": 0,
    "At Risk": 1,
    "Delayed": 2
}

y = y.map(label_map)

y.value_counts()


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model_lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model_lr.fit(X_train, y_train)

y_pred_lr = model_lr.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred_lr)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Logistic Regression Confusion Matrix")
plt.show()


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

model_rf.fit(X_train, y_train)

y_pred_rf = model_rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


In [ ]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf)
    ]
})

comparison


Final Model Selected: Random Forest

Reason:
After removing target leakage, Random Forest demonstrated superior performance compared to Logistic Regression by effectively capturing non-linear interactions between delivery distance, traffic conditions, and weather severity.

